Day 8 – Vertex AI Matching Engine (Vector Search) \n\nThis notebook demonstrates how to:\n1. Load text from GCS and generate embeddings using Vertex AI\n2. Create a Matching Engine index and endpoint\n3. Deploy the index, upsert vectors, and run similarity search

# Vertex AI – Platform Overview

## What is Vertex AI?
**Vertex AI** is Google Cloud's unified, fully-managed machine learning platform that brings together all Google Cloud ML services under one roof. It allows you to:

- **Build** ML models using AutoML or custom training
- **Deploy** models to production at scale
- **Manage** the full ML lifecycle (data, experiments, pipelines, monitoring)
- **Use foundation models** (Gemini, PaLM, Codey, Imagen, etc.) via APIs

---

## Key Components of Vertex AI

### 1. Vertex AI Studio
**Vertex AI Studio** is an interactive, no-code / low-code UI inside the Google Cloud Console that lets you:

| Feature | Description |
|--------|-------------|
| **Prompt Design** | Test and iterate prompts for Gemini & PaLM models in real time |
| **Chat & Completion** | Explore text generation, chat, and multimodal capabilities |
| **Tuning** | Fine-tune foundation models (supervised tuning / RLHF) on your own data |
| **Evaluate** | Evaluate model responses using built-in metrics |
| **Multimodal** | Work with text, images, video, audio inputs together |
| **Code** | Generate, complete, and explain code using Codey models |

> **Use case:** Quickly prototype GenAI applications without writing code before productionising via the SDK.

---

### 2. Vertex AI Search (formerly Enterprise Search)
**Vertex AI Search** is a managed search service powered by Google's search technology and foundation models. It provides:

| Feature | Description |
|--------|-------------|
| **Semantic Search** | Understands meaning, not just keywords — returns relevant results |
| **Unstructured Search** | Index PDFs, HTML, DOCX, and other document types |
| **Structured Search** | Search over BigQuery tables or structured data |
| **Website Search** | Crawl and index your website for internal search |
| **Conversational Search** | Multi-turn search with natural language follow-ups |
| **Answer Generation** | Grounded, citation-backed answers using Gemini |

**How it works:**
1. Create a **Data Store** → ingest documents (GCS, BigQuery, website)
2. Create a **Search App** → configure ranking and serving settings
3. Query via the **Search API** or embed in your application

> **Use case:** Enterprise knowledge bases, customer support portals, e-commerce product search.

---

### 3. Vertex AI RAG Engine
**RAG Engine** (Retrieval-Augmented Generation) is a fully-managed service in Vertex AI that combines retrieval with LLM generation to produce grounded, accurate responses.

#### How RAG Works:
```
User Query
    ↓
Embedding Model  →  Vector Search (retrieve top-k chunks)
    ↓
Retrieved Context + Query
    ↓
LLM (Gemini)  →  Grounded Answer
```

| Component | Role |
|-----------|------|
| **RAG Corpus** | A collection of documents ingested and chunked automatically |
| **Embedding Model** | Converts text chunks into dense vectors (e.g., `text-embedding-005`) |
| **Vector Store** | Managed vector database backed by Vertex AI Matching Engine |
| **Retrieval** | Finds the most semantically relevant chunks for a query |
| **Generation** | Gemini uses retrieved context to generate a grounded answer |

**Key advantages over vanilla LLMs:**
- Reduces **hallucinations** by grounding answers in your data
- Keeps knowledge **up-to-date** without retraining
- Provides **citations** to source documents

> **Use case:** Internal Q&A bots, document summarization, compliance assistants.

---

### 4. Vertex AI Matching Engine (Vector Search)
**Matching Engine** is Google Cloud's high-scale, low-latency **Approximate Nearest Neighbor (ANN)** service — the vector database powering semantic search and RAG.

| Feature | Description |
|--------|-------------|
| **Algorithm** | Tree-AH (Asymmetric Hashing) or ScaNN for fast ANN search |
| **Scale** | Billions of vectors with sub-10ms latency |
| **Index Types** | Batch update or stream update |
| **Distance Metrics** | Cosine, Dot Product, L2 (Euclidean) |
| **Deployment** | Index deployed to an endpoint (public or private VPC) |

**Workflow:**
1. Generate embeddings → store as `IndexDatapoint` objects
2. Create an **Index** with chosen ANN config
3. Create an **IndexEndpoint** and deploy the index
4. Call `find_neighbors()` to retrieve nearest vectors

> This notebook demonstrates exactly this workflow — building a semantic search engine over a Terms of Service document.

---

In [ ]:
# pip install google-cloud-aiplatform

from google.cloud import aiplatform
from vertexai.language_models import TextEmbeddingModel
from google.cloud.aiplatform_v1.types import index as index_types
import uuid

#  1. Init Vertex AI
PROJECT_ID = "gen-lang-client-0384449393"
LOCATION = "us-central1"
aiplatform.init(project=PROJECT_ID, location=LOCATION)

#  2. Load text file from GCS
from google.cloud import storage

BUCKET_NAME = "fictbank_01"
BLOB_NAME = "fictional_bank_terms.txt"

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(BLOB_NAME)
text_data = blob.download_as_text()

NotFound: 404 GET https://storage.googleapis.com/download/storage/v1/b/wibank_bucket/o/terms_of_services.txt?alt=media: The specified bucket does not exist.: ('Request failed with status code', 404, 'Expected one of', <HTTPStatus.OK: 200>, <HTTPStatus.PARTIAL_CONTENT: 206>)

In [ ]:
# Split into small chunks (for embeddings)
documents = [line.strip() for line in text_data.split("\n") if line.strip()]

#  3. Create embeddings
embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-005")

embeddings = [
    embedding_model.get_embeddings([doc])[0].values for doc in documents
]

print(f"Created {len(embeddings)} embeddings of dimension {len(embeddings[0])}")

In [ ]:
from google.cloud.aiplatform.matching_engine import matching_engine_index_config
import uuid

index_id = f"my-index-{uuid.uuid4().hex[:6]}"
print(index_id)

# Create Tree-AH config with proper imports
algorithm_config = matching_engine_index_config.TreeAhConfig(
    leaf_node_embedding_count=500,
    leaf_nodes_to_search_percent=7,
)

config = matching_engine_index_config.MatchingEngineIndexConfig(
    dimensions=len(embeddings[0]),
    algorithm_config=algorithm_config,
    approximate_neighbors_count=10,
    distance_measure_type="COSINE_DISTANCE",
    shard_size="SHARD_SIZE_SMALL",
)

index = aiplatform.MatchingEngineIndex._create(
    display_name=index_id,
    config=config,
    project=PROJECT_ID,
    location=LOCATION,
    index_update_method="STREAM_UPDATE",  # Add this line!
    sync=True,
)

endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name=f"{index_id}-endpoint",
    public_endpoint_enabled=True,   # you can set False for private
)

In [ ]:
endpoint.deploy_index(index=index, deployed_index_id="my_unique_deployment_id_v3")

In [ ]:
datapoints = [
    index_types.IndexDatapoint(
        datapoint_id=str(i),
        feature_vector=emb,
    )
    for i, emb in enumerate(embeddings)
]

index.upsert_datapoints(datapoints=datapoints)
print(f"Upserted {len(datapoints)} vectors ")

In [ ]:
#  1. Build query embedding
query_text = "How to apply for loan"
query_emb = embedding_model.get_embeddings([query_text])[0].values

#  2. Call find_neighbors
neighbors_list = endpoint.find_neighbors(
    deployed_index_id=endpoint.deployed_indexes[0].id,
    queries=[query_emb],
    num_neighbors=5,
)

print("\n Query Results:")
for neighbor in neighbors_list[0]:  # first query
    idx = int(neighbor.id)  #  vector ID
    print(f"- {documents[idx]} (score={neighbor.distance:.4f})")